In [1]:
import sys
sys.path.append("/DATA/srajat/table_qa/script")

In [2]:
import os
import torch
from utils import load_checkpoint_for_eval
from table_matching_model import strubert_embd
import numpy as np
import os

/home/srajat/anaconda3/envs/strubert/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/srajat/anaconda3/envs/strubert/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/srajat/anaconda3/envs/strubert/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/srajat/anaconda3/envs/strubert/lib

In [3]:
os.environ['http_proxy']="http://proxy61.iitd.ac.in:3128"
os.environ['https_proxy']="http://proxy61.iitd.ac.in:3128"

## Creating the dataset for processing BIRD dataset
### Processing the query tables

In [4]:
# model parameters
run_number = "without_nan_with_space_remv_p2_n2_run_1_balanced_2"
m_min_tables = 10
m_min_rows = 8
m_max_rows = 12
m_min_cols = 10
m_max_cols = 25

# corpus parameters
split_number = "without_nan_with_space_remv_run_1"
min_tables = 10
min_rows = 8
max_rows = 12
min_cols = 10
max_cols = 25

In [5]:
m_parm_setting = f'mint_{m_min_tables}_minr_{m_min_rows}_maxr_{m_max_rows}_minc_{m_min_cols}_maxc_{m_max_cols}'
parm_setting = f'mint_{min_tables}_minr_{min_rows}_maxr_{max_rows}_minc_{min_cols}_maxc_{max_cols}'
model_setting = "batch_4_epoch_8_lr_3e-05"

query_path_output = "../../dataset/spider/processing/query/gemini-2.5-flash"
# query_path = "../../dataset/bird/query_tables"

result_path = f"../../output/spider_res/default_{model_setting}_{run_number}_{m_parm_setting}_{parm_setting}_{split_number}/gemini-2.5-flash/"

corpus_path_output = f"../../dataset/spider/processing/corpus_split/{split_number}_{parm_setting}"
# corpus_path = "../../dataset/bird/tables"

corpus_folder = corpus_path_output
query_folder = query_path_output

In [6]:
# # checking if the folder exist in that path

# if not os.path.exists(query_path_output):
#     os.makedirs(query_path_output)
#     print("folder created...")
# else:
#     print("folder already exist...")


In [7]:
# import shutil
# import os

# # renaming the json
# # the json should start with 're_tables-' + file_number + '.json'
# # the table should be name with 'table-' + file_number + '-' + table_number

# files = []

# # checking the number of files in query_tables path
# if os.path.exists(query_path) and os.path.isdir(query_path):
#     files = [f for f in os.listdir(query_path) if os.path.isfile(os.path.join(query_path, f))]
#     print(files)
# else:
#     print(f"Folder not found at {query_path}")

# # Now copying these files to query_and_corpus folder
# if len(files) != 0:
#     for f in files:
#         name = f.strip().split(".")[0].strip().split("_")[1]
#         shutil.copyfile(os.path.join(query_path,f), os.path.join(query_path_output,f"re_tables-{name}.json"))
 

In [8]:
# import json

# # Now preprocessing the tables
# files = [f for f in os.listdir(query_path_output) if os.path.isfile(os.path.join(query_path_output, f))]

# for f in files:
#     # print(f)
#     with open(os.path.join(query_path_output,f), 'r') as fp:
#         data = json.load(fp)

#     data2 = {}
#     data2[f'table-{f.strip().split(".")[0].split("-")[1]}-{f.strip().split(".")[0].split("-")[1]}'] = data[f'table_{f.strip().split(".")[0].split("-")[1]}']

#     with open(os.path.join(query_path_output,f),'w') as fp:
#         json.dump(data2, fp, indent=4)

# print("Processing of the queries completed...")

### Processing the corpus tables

In [9]:
# # checking if the folder exist in that path

# if not os.path.exists(corpus_path_output):
#     os.makedirs(corpus_path_output)
#     print("folder created...")
# else:
#     print("folder already exist...")

In [10]:
# import shutil
# import os
# import json
# import random

# # renaming the json
# # the json should start with 're_tables-' + file_number + '.json'
# # the table should be name with 'table-' + file_number + '-' + table_number

# files_corpus = []

# # checking the number of files in query_tables path
# if os.path.exists(corpus_path) and os.path.isdir(corpus_path):
#     files_corpus = [f for f in os.listdir(corpus_path) if os.path.isfile(os.path.join(corpus_path, f))]
#     print(files_corpus)
# else:
#     print(f"Folder not found at {corpus_path}")

# # # Now copying these files to query_and_corpus folder
# # if len(files_corpus) != 0:
# #     for f in files_corpus:
# #         name = f.strip().split(".")[0]
# #         shutil.copyfile(os.path.join(corpus_path,f), os.path.join(corpus_path_output,f"re_tables-{name}.json"))

# # Read the data
# if len(files_corpus) != 0:
#     for f in files_corpus:
#         # name of the file
#         name = f.strip().split(".")[0]

#         new_file_data = {}
        
#         with open(os.path.join(corpus_path,f), 'r') as fp:
#             json_data = json.load(fp)
#         data_keys = json_data.keys()

#         for k in data_keys:
#             print(k)
#             chunk_size = 10

#             # Shuffle the data randomly
#             data = json_data[k]["data"].copy()
#             random.shuffle(data)

#             i = 0
#             idx = 1
#             limit_tables = 2

#             # If the data length is 0 then while loop will not work
#             # In this case we need to fill the data with empty table only once
#             if len(data) == 0:
#                 new_table = json_data[k].copy()
#                 new_table["data"] = data[i:min(i+chunk_size,len(data))]
#                 new_table["numDataRows"] = len(new_table["data"])
#                 new_file_data[f'{k}__{idx}'] = new_table
                
#             # if the length is more than 0 then it will split into <= limit_tables
#             while(i<len(data)):
#                 new_table = json_data[k].copy()
#                 new_table["data"] = data[i:min(i+chunk_size,len(data))]
#                 new_table["numDataRows"] = len(new_table["data"])

#                 new_file_data[f'{k}__{idx}'] = new_table
#                 idx+=1
#                 i+=chunk_size

#                 if idx>limit_tables:
#                     break
        
#         # writing this in the new location
#         with open(os.path.join(corpus_path_output,f"re_tables-{name}.json"), "w") as f:
#             json.dump(new_file_data, f, indent=2)

# print("Processing completed")

## Loading the Model

In [11]:
# Load the best model from the database
# save_path_best = f'../../output/finetune_model/spider/{model_setting}_{run_number}_{m_parm_setting}/best_spider_tabert_1.pt'
tabert_path = "../../model/tabert_base_k3/model.bin"
device = "cuda:0"

In [12]:
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Device Count:", torch.cuda.device_count())
print("Current CUDA Device:", torch.cuda.current_device())

CUDA Available: True
CUDA Device Count: 4
Current CUDA Device: 0


In [13]:
# Loading the model
model = strubert_embd(tabert_path=tabert_path, device=device) 

In [14]:
# Assume this is your loaded model

# model = load_checkpoint_for_eval(model, save_path_best, device=device)
model.eval()

strubert_embd(
  (embedding): VerticalAttentionTableBert(
    (_bert_model): BertForMaskedLM(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): BertLayerNorm()
          (dropout): Dropout(p=0.1)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0): BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1)
                )
                (output): BertSelfOutput(
                  (dense): Linear(in_features=768, out_features=768,

## Reading the dataset to generate embedding

In [15]:
import os
import json

# creating the list of all the tables

# input will be a list of list [['table-0312-73' 'table-0312-60' '1']]
corpus = []

# List all files
for filename in os.listdir(corpus_folder):
    file_path = os.path.join(corpus_folder, filename)
    if os.path.isfile(file_path):
        print("Reading:", file_path)
        with open(file_path, "r") as f:
            data = json.load(f)
            data_keys = list(data.keys())
            # print(data_keys)
            for k in data_keys:
                corpus.append([k,k,1])

Reading: ../../dataset/spider/processing/corpus_split/without_nan_with_space_remv_run_1_mint_10_minr_8_maxr_12_minc_10_maxc_25/re_tables-party_people.json
Reading: ../../dataset/spider/processing/corpus_split/without_nan_with_space_remv_run_1_mint_10_minr_8_maxr_12_minc_10_maxc_25/re_tables-architecture.json
Reading: ../../dataset/spider/processing/corpus_split/without_nan_with_space_remv_run_1_mint_10_minr_8_maxr_12_minc_10_maxc_25/re_tables-chinook_1.json
Reading: ../../dataset/spider/processing/corpus_split/without_nan_with_space_remv_run_1_mint_10_minr_8_maxr_12_minc_10_maxc_25/re_tables-phone_market.json
Reading: ../../dataset/spider/processing/corpus_split/without_nan_with_space_remv_run_1_mint_10_minr_8_maxr_12_minc_10_maxc_25/re_tables-flight_company.json
Reading: ../../dataset/spider/processing/corpus_split/without_nan_with_space_remv_run_1_mint_10_minr_8_maxr_12_minc_10_maxc_25/re_tables-swimming.json
Reading: ../../dataset/spider/processing/corpus_split/without_nan_with_spac

In [16]:
print(len(corpus))

# corpus = corpus[:30]

5956


In [17]:
from data_reader import DataAndQueryReader
from torch.utils.data import DataLoader
from utils import pad_table_query

batch_size = 5

corpus_dataset = DataAndQueryReader(corpus,corpus_folder)
test_iter = DataLoader(corpus_dataset, batch_size=batch_size, shuffle=False, collate_fn=pad_table_query)


In [18]:
corpus_embddding = []

for i, batch in enumerate(test_iter):
    tables, queries, all_tables_meta, all_query_meta, labels = batch

    labels = torch.FloatTensor(labels).to(device)
    
    with torch.no_grad():  # ensures no gradients are tracked
        outputs = model(tables, all_tables_meta, queries, all_query_meta)

    # detach and move to CPU before storing
    corpus_embddding.append(outputs)

In [19]:
print(len(corpus_embddding))
print(corpus_embddding[-1][0].shape[0])

1192
1


### Reading the query table and generate its embedding

In [20]:
import os
import json

# creating the list of all the tables

# input will be a list of list [['table-0312-73' 'table-0312-60' '1']]
query = []

# List all files
for filename in os.listdir(query_folder):
    file_path = os.path.join(query_folder, filename)
    if os.path.isfile(file_path):
        print("Reading:", file_path)
        with open(file_path, "r") as f:
            data = json.load(f)
            data_keys = list(data.keys())
            # print(data_keys)
            for k in data_keys:
                for _ in range(batch_size):
                    query.append([k,k,1])

Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-602.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-68.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-573.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-24.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-51.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-173.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-113.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-523.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-204.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-260.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-288.json
Reading: ../../dataset/spider/processing/query/gemini-2.5-flash/re_tables-338.json
Reading

In [21]:
print(len(query))

# query = query[:50]
# query

3290


In [22]:
from data_reader import DataAndQueryReader
from torch.utils.data import DataLoader
from utils import pad_table_query

query_dataset = DataAndQueryReader(query,query_folder)

query_iter = DataLoader(query_dataset, batch_size=batch_size, shuffle=False, collate_fn=pad_table_query)

In [23]:
# print(query[1090])
# print(query[1135])
# print(query[1140])
# print(query[1330])


# for bird dataset
# print(query[5955])

In [24]:
query_embddding = []

for i, batch in enumerate(query_iter):
    tables, queries, all_tables_meta, all_query_meta, labels = batch

    labels = torch.FloatTensor(labels).to(device)
    
    with torch.no_grad():  # ensures no gradients are tracked
        outputs = model(tables, all_tables_meta, queries, all_query_meta)

    # detach and move to CPU before storing
    query_embddding.append(outputs)

In [25]:
print(query_embddding[-1][0][:4].shape)

torch.Size([4, 43, 768])


### Get the similarity for query with corpus

In [26]:

res = []

for q in range(len(query_embddding)):
    r = []
    # print(q)
    for c in range(len(corpus_embddding)):
        # print(c)
        with torch.no_grad():  # ensures no gradients are tracked
            # rnk = model.get_similarity_custom(query_embddding[q][0][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][1][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][2][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][3][:min(batch_size, corpus_embddding[c][0].shape[0])], corpus_embddding[c][0], corpus_embddding[c][1], corpus_embddding[c][2], corpus_embddding[c][3])
            rnk = model.get_similarity_cls(query_embddding[q][0][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][1][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][2][:min(batch_size, corpus_embddding[c][0].shape[0])], query_embddding[q][3][:min(batch_size, corpus_embddding[c][0].shape[0])], corpus_embddding[c][0], corpus_embddding[c][1], corpus_embddding[c][2], corpus_embddding[c][3])
        r.append(rnk)
    res.append(r)

In [27]:
len(res[0][0])

5

In [28]:
print(res[0])

[tensor([0.5129, 0.5047, 0.4408, 0.4502, 0.4666], device='cuda:0'), tensor([0.4541, 0.4598, 0.4680, 0.4665, 0.4638], device='cuda:0'), tensor([0.4731, 0.4712, 0.4626, 0.4662, 0.6110], device='cuda:0'), tensor([0.6046, 0.5823, 0.5872, 0.5659, 0.5665], device='cuda:0'), tensor([0.5604, 0.5645, 0.5708, 0.5694, 0.5697], device='cuda:0'), tensor([0.5694, 0.5648, 0.5641, 0.4984, 0.4995], device='cuda:0'), tensor([0.6006, 0.5966, 0.5967, 0.6151, 0.5997], device='cuda:0'), tensor([0.6071, 0.6000, 0.6008, 0.5999, 0.5989], device='cuda:0'), tensor([0.6632, 0.6559, 0.6576, 0.6536, 0.6488], device='cuda:0'), tensor([0.6561, 0.6601, 0.6569, 0.6536, 0.6659], device='cuda:0'), tensor([0.6403, 0.6085, 0.6259, 0.6333, 0.6472], device='cuda:0'), tensor([0.6314, 0.6504, 0.6225, 0.6306, 0.6202], device='cuda:0'), tensor([0.6359, 0.6355, 0.6335, 0.6497, 0.6318], device='cuda:0'), tensor([0.5841, 0.6027, 0.5952, 0.5831, 0.5933], device='cuda:0'), tensor([0.5936, 0.6073, 0.5933, 0.5976, 0.6061], device='cuda

In [29]:
corpus_list = []

for c in corpus:
    corpus_list.append(c[0])

In [30]:
print(len(corpus_list))

5956


In [31]:
query_list = []
for q in query:
    if q[0] not in query_list:
        query_list.append(q[0])
    

In [32]:
print(len(query_list))

658


In [33]:
# checking if the folder exist in that path

if not os.path.exists(result_path):
    os.makedirs(result_path)
    print("folder created...")
else:
    print("folder already exist...")

folder created...


In [34]:
from torch import nn
m = nn.Sigmoid()

for idx, r in enumerate(res):
    out = torch.cat(r, dim=0)
    out_s = m(out)

    res_dic = {}

    for o in range(out_s.shape[0]):
        if out_s[o].item() not in res_dic.keys():
            res_dic[out_s[o].item()] = []
        res_dic[out_s[o].item()].append(o)

    res_dic_key = list(res_dic.keys())
    sorted_res_dic_key = sorted(res_dic_key, reverse=True)
    
    with open(os.path.join(result_path, query_list[idx]), 'w') as fp:
        for k in sorted_res_dic_key:
            fp.write(f'{k} : ')
            for j in res_dic[k]:
                fp.write(f'{corpus_list[j]} ')
            
            fp.write('\n')

In [35]:
# from data_reader import DataAndQueryReader
# folds_path= "wikitables_similarity_folds.npy"
# data_folder = "../../dataset/spider/processing/query"

# kfolds = np.load(folds_path, allow_pickle=True)
# kfolds = kfolds[()]

# print(kfolds)

# for fold_index,(train, test) in enumerate(kfolds):

#     print("train length:", len(train))
#     print("test length:",len(test))

#     train_d = train[:2]
#     print(train_d)

#     test_d = test[:2]
#     print(test_d)

#     # train_dataset = DataAndQueryReader(train_d,data_folder)
#     # print('size of training = {}'.format(len(train_dataset)))

#     # print(train_dataset)